# 📅 2026-03-22 (일) 개발 노트 : AI 게임 큐레이터 본진 구축 및 엔진 장착

## 🎯 오늘의 목표
- [x] AI 추천 엔진 기초 라이브러리(pandas, scikit-learn, sentence-transformers) 설치
- [x] Next.js 기반 웹 서비스 본진(hidden-gem-web) 프로젝트 생성
- [x] 메인 랜딩 페이지 UI 초안 구현 및 로컬 서버 가동

## 🛠 진행 상황 및 핵심 기록
- **AI 실험실 세팅**: pip로 pandas, scikit-learn, sentence-transformers 설치 완료. 게임 데이터를 벡터화해서 유사도를 계산할 준비를 마침.
- **웹 본진 구축**: npx create-next-app@latest 명령어로 Next.js 프로젝트 생성. 
- **핵심 경로**: src/app/page.tsx 파일에 서비스 메인 UI 코드 이식 완료.
- **사용 도구**: Lucide-react(아이콘), Tailwind CSS(디자인), Node.js(런타임).

## 🚨 트러블슈팅 (문제 및 해결)
- **문제 1: npx 커맨드 인식 불가**
    - **원인**: 시스템에 Node.js 엔진이 설치되지 않음.
    - **해결**: Node.js 공식 홈페이지에서 LTS 버전 설치 후 터미널 재실행하여 해결.
- **문제 2: 폴더명 대문자 에러**
    - **원인**: npm 패키지 명명 규칙상 프로젝트 이름에 대문자 불가.
    - **해결**: hidden-gem-web으로 소문자 폴더명을 사용하여 재생성.
- **문제 3: 아이콘 라이브러리 미설치**
    - **원인**: 코드에는 아이콘을 썼지만 실제 패키지가 본진 폴더에 없었음.
    - **해결**: npm install lucide-react 실행 후 정상 출력 확인.

## 💡 인사이트 및 다음 할 일
- **배운 점**: 
    - 깃허브에는 설계도만 올리고, 무거운 부품(node_modules)은 새 자리에서 npm install로 조달함.
    - AI 모델이 문장을 숫자로 바꾸고 거리를 계산해서 추천한다는 원리 파악.
- **다음 할 일**: 
    - 오늘 만든 검색창에 실제 500개 게임 CSV 데이터 연결하기.
    - 유저 입력값과 게임 데이터를 비교하는 백엔드 API 로직 작성.

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# 1. find_dotenv()가 마법의 핵심입니다! 상위 폴더를 뒤져서 .env를 찾아냅니다.
load_dotenv(find_dotenv())
API_KEY = os.getenv("RAWG_API_KEY")

if not API_KEY:
    print("🚨 에러: API 키를 찾을 수 없습니다. .env 파일 위치를 다시 확인해주세요!")
else:
    print("✅ 최상위 폴더에서 API 키 로드 완벽 성공! 💯 데이터를 가져옵니다...")

    # 2. RAWG API로 평점(rating) 높은 순으로 게임 40개 가져오기
    url = f"https://api.rawg.io/api/games?key={API_KEY}&page_size=40&ordering=-rating"
    response = requests.get(url).json()

    games_data = []
    # 3. 데이터 속에서 추천 시스템에 쓸 핵심 알맹이만 쏙쏙 빼옵니다.
    for game in response.get('results', []):
        games_data.append({
            'id': game['id'],
            'name': game['name'],
            'rating': game['rating'], # 평점
            'ratings_count': game['ratings_count'], # 리뷰 수 (인지도 파악용)
            'genres': ", ".join([g['name'] for g in game['genres']]), # 장르들 합치기
            'tags': ", ".join([t['name'] for t in game['tags'][:5]])  # 핵심 태그 5개만 합치기
        })

    # 4. 보기 좋은 표(데이터프레임)로 변환
    df_games = pd.DataFrame(games_data)

    # 5. 나중을 위해 CSV 파일로 백업 저장 (현재 작업 중인 26-03 폴더 안에 저장됩니다)
    df_games.to_csv("hidden_gem_raw_data.csv", index=False)
    print("🎉 hidden_gem_raw_data.csv 파일로 백업 완료!")

    # 6. 결과 확인 (display를 쓰면 표가 아주 예쁘게 나옵니다)
    display(df_games.head())

✅ 최상위 폴더에서 API 키 로드 완벽 성공! 💯 데이터를 가져옵니다...
🎉 hidden_gem_raw_data.csv 파일로 백업 완료!


,id,name,rating,ratings_count,genres,tags
0,58781,The Elder Scrolls VI,4.86,6,"Action, RPG","Singleplayer, Atmospheric, RPG, Open World, Fi..."
1,975381,No Case Should Remain Unsolved,4.83,6,"Adventure, Indie","Singleplayer, Steam Achievements, Steam Cloud,..."
2,53962,Gimmick!,4.83,5,,Singleplayer
3,53594,Super Robot Taisen: Original Generation,4.83,6,"Action, Strategy","Singleplayer, exclusive, true exclusive"
4,43252,The Witcher 3: Wild Hunt – Blood and Wine,4.81,627,RPG,"Horror, War, Blood, love"


In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# 1. 컴퓨터가 .env를 도대체 어디서 찾았는지 경로를 출력해 봅니다.
env_path = find_dotenv()
print(f"🔍 찾은 .env 파일 경로: '{env_path}'")

# 2. 주피터 노트북의 예전 기억을 강제로 지우고 새로 덮어씌웁니다 (override=True)
load_dotenv(env_path, override=True)

# 3. 키가 제대로 들어왔는지 확인합니다.
key = os.getenv("RAWG_API_KEY")

if key:
    print(f"✅ 성공! 키 값 앞자리 확인: {key[:5]}...")
else:
    print("🚨 여전히 못 찾았습니다. .env 파일 내부를 확인해야 합니다!")

🔍 찾은 .env 파일 경로: 'c:\div_log_kjt\hidden-gem-devlog\.env'
✅ 성공! 키 값 앞자리 확인: 75dcf...


In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers pandas requests python-dotenv

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/512.4 kB ? eta -:--:--
   --------------------------------------- 512.4/512.4 kB 16.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/618.0 kB ? eta -:--:--
   --------------------------------------- 618.0/618.0 kB 38.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   --------- ------------------------------ 2.5/10.7 MB 52.1 MB/s eta 0:00:01
   --------------- ------------------------ 4.1/10.7 MB 38.0 MB/s eta 0:00:01
   --------------------------- ------------ 7.5/10.7 MB 52.8 MB/s eta 0:00:01
   ------------------------------ --------- 8.2/10.7 MB 47.4 MB/s eta 0:00:01
   ------------------------------------ --- 9.6/10.7 MB 41.0 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install --user --upgrade sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

# 1. 수집했던 데이터 불러오기
df = pd.read_csv("hidden_gem_raw_data.csv")
df = df.fillna("")

# 2. AI가 읽기 좋게 데이터 합치기
df['combined_text'] = "Game: " + df['name'] + ". Genres: " + df['genres'] + ". Tags: " + df['tags']

# 3. 모델 로드 (가장 가볍고 성능 좋은 모델)
print("🤖 AI 모델 로딩 중...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 4. 텍스트를 벡터(숫자)로 변환
print("✨ 텍스트 데이터를 숫자로 변환하고 있습니다. 잠시만 기다려주세요...")
embeddings = model.encode(df['combined_text'].tolist())

print("\n🎉 임베딩 변환 완료!")
print(f"변환된 데이터 크기: {embeddings.shape}") # (40, 384)가 나오면 성공!

🤖 AI 모델 로딩 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✨ 텍스트 데이터를 숫자로 변환하고 있습니다. 잠시만 기다려주세요...

🎉 임베딩 변환 완료!
변환된 데이터 크기: (40, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. 유저의 가상 질문 (이 문구를 바꿔가며 테스트해보세요!)
# 예: "I want a dark fantasy RPG with a great story" 
# 또는 "Simple indie adventure game"
user_query = "I'm looking for an open world RPG with epic story like Witcher"

# 2. 유저 질문도 AI 모델을 통해 384개의 숫자로 변환합니다.
query_embedding = model.encode([user_query])

# 3. 코사인 유사도 계산 (질문과 40개 게임 사이의 '거리'를 측정)
# 1에 가까울수록 아주 비슷한 게임입니다.
similarities = cosine_similarity(query_embedding, embeddings).flatten()

# 4. 원본 데이터에 유사도 점수를 붙여서 정렬
df['similarity'] = similarities
recommendations = df.sort_values(by='similarity', ascending=False)

print(f"🔎 내 취향 입력: '{user_query}'")
print("-" * 50)
print("🎯 AI가 추천하는 내 취향 저격 TOP 5:")

# 깔끔하게 이름, 유사도 점수, 평점만 출력해봅시다.
display(recommendations[['name', 'similarity', 'rating', 'genres']].head(5))

🔎 내 취향 입력: 'I'm looking for an open world RPG with epic story like Witcher'
--------------------------------------------------
🎯 AI가 추천하는 내 취향 저격 TOP 5:


,name,similarity,rating,genres
5,The Witcher 3 Wild Hunt - Complete Edition,0.564436,4.80,"Adventure, RPG"
4,The Witcher 3: Wild Hunt – Blood and Wine,0.543725,4.81,RPG
6,The Witcher 3: Wild Hunt – Hearts of Stone,0.537766,4.76,RPG
0,The Elder Scrolls VI,0.495523,4.86,"Action, RPG"
34,Shining Force III,0.466813,4.67,"RPG, Strategy"


In [8]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv, find_dotenv

# 1. 메모리에서 날아간 API 키 다시 불러오기 (매우 중요!)
load_dotenv(find_dotenv(), override=True)
API_KEY = os.getenv("RAWG_API_KEY")

if not API_KEY:
    print("🚨 여전히 API 키를 못 찾았습니다! .env 파일을 확인해주세요.")
else:
    all_games_data = []
    pages_to_fetch = 12 # 5070 사양이면 20페이지도 껌이지만 일단 12페이지(480개) 고!

    print(f"🚀 총 {pages_to_fetch * 40}개의 게임 데이터를 수집 시작합니다...")

    for page in range(1, pages_to_fetch + 1):
        url = f"https://api.rawg.io/api/games?key={API_KEY}&page_size=40&page={page}&ordering=-rating"
        try:
            response = requests.get(url).json()
            
            for game in response.get('results', []):
                all_games_data.append({
                    'name': game['name'],
                    'rating': game['rating'],
                    'genres': ", ".join([g['name'] for g in game['genres']]),
                    'tags': ", ".join([t['name'] for t in game['tags'][:5]])
                })
            
            print(f"📦 {page}페이지 수집 완료... (누적 {len(all_games_data)}개)")
            time.sleep(0.1) # 5070 속도를 API 서버가 못 따라올 수 있으니 살짝 매너 타임
        except Exception as e:
            print(f"⚠️ {page}페이지에서 오류 발생: {e}")

    # 데이터프레임 변환 및 저장
    df_bulk = pd.DataFrame(all_games_data)
    df_bulk.to_csv("hidden_gem_bulk_data.csv", index=False)

    print(f"\n✅ 수집 종료! 총 {len(df_bulk)}개의 데이터가 'hidden_gem_bulk_data.csv'에 저장되었습니다.")

🚀 총 480개의 게임 데이터를 수집 시작합니다...
📦 1페이지 수집 완료... (누적 40개)
📦 2페이지 수집 완료... (누적 80개)
📦 3페이지 수집 완료... (누적 120개)
📦 4페이지 수집 완료... (누적 160개)
📦 5페이지 수집 완료... (누적 200개)
📦 6페이지 수집 완료... (누적 240개)
📦 7페이지 수집 완료... (누적 280개)
📦 8페이지 수집 완료... (누적 320개)
📦 9페이지 수집 완료... (누적 360개)
📦 10페이지 수집 완료... (누적 400개)
📦 11페이지 수집 완료... (누적 440개)
📦 12페이지 수집 완료... (누적 480개)

✅ 수집 종료! 총 480개의 데이터가 'hidden_gem_bulk_data.csv'에 저장되었습니다.


In [9]:
# 1. 벌크 데이터용 통합 텍스트 만들기
df_bulk['combined_text'] = "Game: " + df_bulk['name'] + ". Genres: " + df_bulk['genres'] + ". Tags: " + df_bulk['tags']

# 2. 임베딩 실행 (약 500개 데이터를 숫자로 변환)
print("✨ 783D 가동! 대량 데이터 임베딩 중...")
bulk_embeddings = model.encode(df_bulk['combined_text'].tolist(), show_progress_bar=True)

# 3. 유저 취향 검색 (이번엔 좀 더 구체적으로 적어보세요!)
user_query = "I want a high-quality open world game with a deep story and fantasy elements"
query_embedding = model.encode([user_query])

# 4. 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity
bulk_similarities = cosine_similarity(query_embedding, bulk_embeddings).flatten()

# 5. 결과 출력
df_bulk['similarity'] = bulk_similarities
top_recommendations = df_bulk.sort_values(by='similarity', ascending=False)

print(f"\n🔎 검색어: '{user_query}'")
print("-" * 50)
print("🎯 500개 데이터 중 AI가 선별한 TOP 10:")
display(top_recommendations[['name', 'similarity', 'rating', 'genres']].head(10))

✨ 783D 가동! 대량 데이터 임베딩 중...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]


🔎 검색어: 'I want a high-quality open world game with a deep story and fantasy elements'
--------------------------------------------------
🎯 500개 데이터 중 AI가 선별한 TOP 10:


,name,similarity,rating,genres
292,Dragon Age: Origins Awakening,0.586793,4.46,RPG
472,Ghost of Tsushima,0.556879,4.41,"Action, Adventure, RPG"
196,Granblue Fantasy,0.554662,4.50,RPG
83,The Legend of Zelda: Twilight Princess HD,0.554488,4.58,"Action, RPG"
330,Dragon Age: Origins - Ultimate Edition,0.554056,4.44,RPG
202,Shining Force: Resurrection of the Dark Dragon,0.552008,4.50,"RPG, Strategy"
162,Beyond Good & Evil - 20th Anniversary Edition,0.546720,4.50,"Action, Adventure"
113,The Legend of Zelda: The Wind Waker HD,0.544915,4.57,Adventure
364,Hauntii,0.541779,4.43,"Action, Adventure, Indie"
44,Attack the Light - Steven Universe Light RPG,0.541606,4.67,"Adventure, RPG"


In [10]:
#한글 번역기 
import sys
!{sys.executable} -m pip install deep_translator


   ---------------------------------------- 0.0/42.3 kB ? eta -:--:--
   ---------------------------------------- 42.3/42.3 kB 1.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from deep_translator import GoogleTranslator
from sklearn.metrics.pairwise import cosine_similarity

# 1. 783D 오너의 한글 취향 입력!
korean_query = "나는 위쳐처럼 스토리가 깊고 판타지 세계관인 오픈월드 액션 RPG를 찾고 있어"

# 2. 한글 -> 영어 자동 번역 (AI 모델이 영어를 좋아하니까요!)
english_query = GoogleTranslator(source='ko', target='en').translate(korean_query)
print(f"🌐 AI용 번역 결과: {english_query}")

# 3. 번역된 검색어로 유사도 계산
query_embedding = model.encode([english_query])
bulk_similarities = cosine_similarity(query_embedding, bulk_embeddings).flatten()

# 4. 결과 출력
df_bulk['similarity'] = bulk_similarities
final_recommendations = df_bulk.sort_values(by='similarity', ascending=False)

print("-" * 50)
print(f"🎯 '{korean_query}' 취향 저격 결과:")
display(final_recommendations[['name', 'similarity', 'rating', 'genres']].head(10))

🌐 AI용 번역 결과: I'm looking for an open world action RPG with a deep story and a fantasy world like The Witcher.
--------------------------------------------------
🎯 '나는 위쳐처럼 스토리가 깊고 판타지 세계관인 오픈월드 액션 RPG를 찾고 있어' 취향 저격 결과:


,name,similarity,rating,genres
5,The Witcher 3 Wild Hunt - Complete Edition,0.620465,4.80,"Adventure, RPG"
50,The Witcher 3: Wild Hunt,0.611375,4.64,"Action, RPG"
292,Dragon Age: Origins Awakening,0.593451,4.46,RPG
4,The Witcher 3: Wild Hunt – Blood and Wine,0.586467,4.81,RPG
472,Ghost of Tsushima,0.574005,4.41,"Action, Adventure, RPG"
6,The Witcher 3: Wild Hunt – Hearts of Stone,0.572079,4.76,RPG
330,Dragon Age: Origins - Ultimate Edition,0.568982,4.44,RPG
364,Hauntii,0.560919,4.43,"Action, Adventure, Indie"
202,Shining Force: Resurrection of the Dark Dragon,0.556991,4.50,"RPG, Strategy"
91,Dungeons of Dreadrock,0.553115,4.57,"Action, Adventure, RPG, Casual, Indie, Puzzle"
